# 04d · Baseline only sanity check (C5)

**Purpose.** Fit the same heads on the baseline part only (before any gas or scent is presented). A fair task scores near chance: 10% for GasWindTunnel multiclass, 25% for EverydayScents multiclass.

**Inputs**
- GasWindTunnel: pre-release segment of each recording, (600, 72, T_base)
- EverydayScents: baseline phase, (144, 11, 60)

**Outputs**
- `results/04d_baseline_sanity_check.csv`

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.raw import raw_features

NAME = "04d_baseline_sanity_check"

## C5: train the same heads on the baseline part only
The baseline is recorded **before** any gas or scent is presented, so a fair task should score near chance
(macro F1 about 1/K: 0.10 for GasWindTunnel multiclass, 0.25 for EverydayScents multiclass).
The baseline is used **without** baseline correction (the resting level is added back), because the resting level
is exactly what could leak the label (recording batches over months, black pepper recorded on 4 days only).

Two baseline-only representations:
- `B1_baseline_raw`: the uncorrected baseline curves, downsampled (R1 on the baseline)
- `B2_baseline_level`: the resting level only, one number per sensor (R2 feature 1)

In [ ]:
parts = []
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)                       # (N, C, T)
    levels = load_baseline_level(dataset)                         # (N, C)
    n_base = info["n_base"]
    X_base = X[:, :, :n_base] + levels[:, :, None]                # (N, C, n_base) uncorrected baseline
    F_b1 = raw_features(X_base, max(2, n_base // 3))              # (N, C * n_base/3)
    F_b2 = levels                                                 # (N, C)
    for task in TASKS:
        for rep, F in [("B1_baseline_raw", F_b1), ("B2_baseline_level", F_b2)]:
            s, _ = run_cv(F, meta, dataset, task, rep, precomputed, verbose=False)
            parts.append(s)
scores = pd.concat(parts, ignore_index=True)
save_results(scores, None, NAME)

In [ ]:
chance = {("gwt", "multi"): 0.10, ("es", "multi"): 0.25, ("gwt", "binary"): 0.5, ("es", "binary"): 0.5}
table = summarize(scores, "balanced_acc")
table["chance (balanced acc)"] = [chance[(d, t)] for d, t, _ in table.index]
print("balanced accuracy (chance level is exact for this metric)")
display(table)
print("macro F1")
display(summarize(scores))

In [ ]:
best = scores.groupby(["dataset", "task", "rep", "head"])["balanced_acc"].mean().groupby(["dataset", "task"]).max()
for (d, t), v in best.items():
    flag = "ABOVE CHANCE: possible leak, report it" if v > chance[(d, t)] + 0.10 else "near chance: ok"
    print(f"{d} {t}: best baseline-only balanced accuracy {v:.3f} (chance {chance[(d, t)]:.2f}) -> {flag}")